In [1]:
%pip install gymnasium numpy

Note: you may need to restart the kernel to use updated packages.


# TribesBridge tests

In [2]:
from tribes_bridge import compile_java, TribesBridge

compile_java()
print("compiled")

compiled


Note: Some input files use unchecked or unsafe operations.
Note: Recompile with -Xlint:unchecked for details.


In [3]:
# Reset from a level file and inspect the returned state

with TribesBridge(compile_first=False) as bridge:
    state = bridge.reset(
        level_file="tribes/levels/SampleLevel.csv",
        game_mode="CAPITALS",
        seed=123,
    )
    print("tick:", state["tick"])
    print("active tribe:", state["activeTribeID"])
    print("game over:", state["gameIsOver"])
    print("actions available:", len(bridge.actions()))
    print("first action:", bridge.actions()[0])

tick: 0
active tribe: 0
game over: False
actions available: 26
first action: {'index': 0, 'text': 'RESEARCH_TECH by tribe 0 : FISHING', 'type': 'RESEARCH_TECH'}


In [4]:
# Step through a visible turn change

with TribesBridge(compile_first=False) as bridge:
    bridge.reset(
        level_file="tribes/levels/SampleLevel.csv",
        game_mode="CAPITALS",
        seed=123,
    )

    actions = bridge.actions()
    end_turn_index = next(
        (i for i, action in enumerate(actions) if action["type"] == "END_TURN"),
        0,
    )

    result = bridge.step(end_turn_index)
    print("tick after step:", result["state"]["tick"])
    print("active tribe after step:", result["state"]["activeTribeID"])
    print("next actions:", len(result["actions"]))

tick after step: 0
active tribe after step: 1
next actions: 24


In [5]:
# Reset from a generated level instead of a file

with TribesBridge(compile_first=False) as bridge:
    state = bridge.reset(
        level_seed=42,
        tribes=["Xin Xi", "Imperius"],
        game_mode="CAPITALS",
        seed=123,
    )
    print("generated tick:", state["tick"])
    print("generated active tribe:", state["activeTribeID"])
    print("actions:", len(bridge.actions()))

generated tick: 0
generated active tribe: 0
actions: 15


In [6]:
# Run a short episode loop

with TribesBridge(compile_first=False) as bridge:
    bridge.reset(
        level_file="tribes/levels/SampleLevel.csv",
        game_mode="CAPITALS",
        seed=123,
    )

    for step_num in range(10):
        state = bridge.observe()
        if state["gameIsOver"]:
            print("finished at step", step_num)
            break

        actions = bridge.actions()
        action_index = next(
            (i for i, action in enumerate(actions) if action["type"] == "END_TURN"),
            0,
        )
        result = bridge.step(action_index)
        print(f"step_num: {step_num}, tick: {result['state']['tick']}, activeTribeID: {result['state']['activeTribeID']}")

step_num: 0, tick: 0, activeTribeID: 1
step_num: 1, tick: 0, activeTribeID: 2
step_num: 2, tick: 0, activeTribeID: 3
step_num: 3, tick: 0, activeTribeID: 0
step_num: 4, tick: 0, activeTribeID: 1
step_num: 5, tick: 0, activeTribeID: 2
step_num: 6, tick: 0, activeTribeID: 3
step_num: 7, tick: 0, activeTribeID: 0
step_num: 8, tick: 0, activeTribeID: 1
step_num: 9, tick: 0, activeTribeID: 2


## TribesEnv proof of concept

The next cells show the Gymnasium wrapper in a simple debugging setup.
It returns the full Tribes game state as JSON and exposes the current legal actions through `action_masks()` and `info["legal_actions"]`.

In [7]:
from tribes_env import TribesEnv
import json

env = TribesEnv(
    level_file="tribes/levels/SampleLevel.csv",
    game_mode="CAPITALS",
    seed=123,
)

def show(obj, limit=900):
    text = json.dumps(obj, indent=2, sort_keys=True)
    print(text[:limit] + ("\n..." if len(text) > limit else ""))

Note: Some input files use unchecked or unsafe operations.
Note: Recompile with -Xlint:unchecked for details.


In [8]:
env.reset()

for step_num in range(5):
    mask = env.action_masks()
    valid_actions = [i for i, allowed in enumerate(mask) if allowed]
    action_index = valid_actions[0]
    obs, reward, terminated, truncated, info = env.step(action_index)
    print(
        f"step={step_num} action={action_index} reward={reward} "
        f"terminated={terminated} truncated={truncated} "
        f"active_tribe={info['active_tribe_id']} tick={info['tick']}"
    )
    if terminated or truncated:
        break

step=0 action=0 reward=100.0 terminated=False truncated=False active_tribe=0 tick=0
step=1 action=0 reward=0.0 terminated=False truncated=False active_tribe=0 tick=0
step=2 action=0 reward=0.0 terminated=False truncated=False active_tribe=1 tick=0
step=3 action=0 reward=100.0 terminated=False truncated=False active_tribe=1 tick=0
step=4 action=0 reward=0.0 terminated=False truncated=False active_tribe=1 tick=0


In [9]:
end_turn_index = next(
    (i for i, action in enumerate(info["legal_actions"]) if action["type"] == "END_TURN"),
    0,
)

obs, reward, terminated, truncated, info = env.step(end_turn_index)
print("Chosen action index:", end_turn_index)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print("Next active tribe:", info["active_tribe_id"])
print("Next tick:", info["tick"])
print("Next legal actions:", len(info["legal_actions"]))

Chosen action index: 0
Reward: 0.0
Terminated: False
Truncated: False
Next active tribe: 2
Next tick: 0
Next legal actions: 25


In [10]:
env = TribesEnv(
    level_file="tribes/levels/SampleLevel.csv",
    game_mode="CAPITALS",
    seed=123,
    max_episode_steps=10,
    compile_first=False,
)

obs, info = env.reset()
print("Observation keys:", obs.keys())
print("JSON length:", len(obs["state_json"]))
print("Active tribe:", info["active_tribe_id"])
print("Tick:", info["tick"])
print("Legal actions:", len(info["legal_actions"]))
print("Action mask true count:", sum(info["action_mask"]))
show(info["legal_actions"][0])

Observation keys: dict_keys(['state_json'])
JSON length: 14138
Active tribe: 0
Tick: 0
Legal actions: 26
Action mask true count: 26
{
  "index": 0,
  "text": "RESEARCH_TECH by tribe 0 : FISHING",
  "type": "RESEARCH_TECH"
}


In [11]:
env.render()

{'state_json': '{"activeTribeID":0,"board":{"actorIDcounter":8,"building":[[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1]],"cityID":[[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1],[-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,1,1,1,-1,-1,-1],[-1,-1,-1,-1,-1,-

In [21]:
env.close()